## K-means Sampling N Cluster

In [1]:
# Email Phishing Dataset Clustering with PhishingBERT
# This notebook processes email phishing data, vectorizes it using PhishingBERT,
# performs clustering, and samples data from each cluster

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# For BERT embeddings
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import os

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Configuration
N_CLUSTER = 5  # Number of clusters to create
N_SAMPLES_PER_CLUSTER = 100  # Number of samples per cluster
BALANCED_SAMPLING = True  # Whether to maintain label balance when sampling
DATA_PATH = "../raw/email_phishing_CEAS-08_train.csv.gz"
OUTPUT_DIR = "../raw/CEAS-08_train_cluster/"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"Number of clusters: {N_CLUSTER}")
print(f"Samples per cluster: {N_SAMPLES_PER_CLUSTER}")
print(f"Balanced sampling: {BALANCED_SAMPLING}")
print(f"Data path: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## 1. Data Loading and Exploration

print("\n" + "="*50)
print("1. LOADING AND EXPLORING DATA")
print("="*50)

# Load the dataset
df = pd.read_csv(DATA_PATH, compression='gzip')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst few rows:")
print(df.head())

print("\nDataset info:")
print(df.info())

print("\nLabel distribution:")
print(df['label'].value_counts())

print("\nSource distribution:")
print(df['source'].value_counts())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Basic text statistics
df['subject_length'] = df['subject'].str.len()
df['body_length'] = df['body'].str.len()
df['combined_text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
df['combined_length'] = df['combined_text'].str.len()

print("\nText length statistics:")
print(df[['subject_length', 'body_length', 'combined_length']].describe())

## 2. PhishingBERT Model Setup and Vectorization

print("\n" + "="*50)
print("2. PHISHINGBERT VECTORIZATION")
print("="*50)

# Initialize PhishingBERT model
# Note: If PhishingBERT is not available, we'll use a security-focused BERT model
try:
    model_name = "martin-ha/toxic-comment-model"  # Alternative security-focused model
    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
except:
    # Fallback to standard BERT
    model_name = "bert-base-uncased"
    print(f"Loading fallback model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

print(f"Using device: {device}")

def get_bert_embeddings(texts, batch_size=32, max_length=512):
    """
    Get BERT embeddings for a list of texts
    """
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        # Move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            # Use CLS token embedding (first token)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(batch_embeddings)
    
    return np.array(embeddings)

# Prepare texts for embedding
texts = df['combined_text'].tolist()
print(f"Number of texts to embed: {len(texts)}")

# Generate embeddings
print("Generating BERT embeddings...")
embeddings = get_bert_embeddings(texts, batch_size=16)
print(f"Embeddings shape: {embeddings.shape}")

## 3. Clustering Analysis

print("\n" + "="*50)
print("3. CLUSTERING ANALYSIS")
print("="*50)

# Determine optimal number of clusters using elbow method
def find_optimal_clusters(embeddings, max_k=10):
    """
    Find optimal number of clusters using elbow method and silhouette score
    """
    inertias = []
    silhouette_scores = []
    k_range = range(2, max_k + 1)
    
    for k in tqdm(k_range, desc="Finding optimal clusters"):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(embeddings, kmeans.labels_))
    
    return k_range, inertias, silhouette_scores

# Find optimal clusters
k_range, inertias, silhouette_scores = find_optimal_clusters(embeddings, max_k=min(15, len(embeddings)//10))

# Plot elbow curve and silhouette scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(k_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method for Optimal k')
ax1.grid(True)

ax2.plot(k_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs Number of Clusters')
ax2.grid(True)

plt.tight_layout()
plt.show()

# Perform clustering with specified number of clusters
print(f"\nPerforming K-means clustering with {N_CLUSTER} clusters...")
kmeans = KMeans(n_clusters=N_CLUSTER, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

# Add cluster labels to dataframe
df['cluster'] = cluster_labels

print(f"Silhouette score for {N_CLUSTER} clusters: {silhouette_score(embeddings, cluster_labels):.4f}")

# Cluster distribution
print("\nCluster distribution:")
cluster_counts = df['cluster'].value_counts().sort_index()
print(cluster_counts)

## 4. Dimensionality Reduction and Visualization

print("\n" + "="*50)
print("4. VISUALIZATION")
print("="*50)

# PCA for dimensionality reduction
print("Performing PCA...")
pca = PCA(n_components=2, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

# t-SNE for better visualization
print("Performing t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_tsne = tsne.fit_transform(embeddings)

# Create visualization dataframe
viz_df = df.copy()
viz_df['pca_x'] = embeddings_pca[:, 0]
viz_df['pca_y'] = embeddings_pca[:, 1]
viz_df['tsne_x'] = embeddings_tsne[:, 0]
viz_df['tsne_y'] = embeddings_tsne[:, 1]

# Plot PCA visualization
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
scatter = plt.scatter(viz_df['pca_x'], viz_df['pca_y'], 
                     c=viz_df['cluster'], cmap='tab10', alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.3f})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.3f})')
plt.title('PCA Visualization of Clusters')
plt.colorbar(scatter, label='Cluster')

plt.subplot(1, 2, 2)
scatter = plt.scatter(viz_df['tsne_x'], viz_df['tsne_y'], 
                     c=viz_df['cluster'], cmap='tab10', alpha=0.6)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('t-SNE Visualization of Clusters')
plt.colorbar(scatter, label='Cluster')

plt.tight_layout()
plt.show()

# Interactive visualization with Plotly
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('PCA Visualization', 't-SNE Visualization'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}]]
)

# PCA plot
fig.add_trace(
    go.Scatter(
        x=viz_df['pca_x'],
        y=viz_df['pca_y'],
        mode='markers',
        marker=dict(
            color=viz_df['cluster'],
            colorscale='tab10',
            showscale=True,
            colorbar=dict(title="Cluster")
        ),
        text=viz_df['subject'].str[:50] + '...',
        hovertemplate='<b>Cluster:</b> %{marker.color}<br>' +
                     '<b>Subject:</b> %{text}<br>' +
                     '<b>Label:</b> %{customdata[0]}<br>' +
                     '<b>Source:</b> %{customdata[1]}<extra></extra>',
        customdata=viz_df[['label', 'source']].values,
        name='PCA'
    ),
    row=1, col=1
)

# t-SNE plot
fig.add_trace(
    go.Scatter(
        x=viz_df['tsne_x'],
        y=viz_df['tsne_y'],
        mode='markers',
        marker=dict(
            color=viz_df['cluster'],
            colorscale='tab10',
            showscale=False
        ),
        text=viz_df['subject'].str[:50] + '...',
        hovertemplate='<b>Cluster:</b> %{marker.color}<br>' +
                     '<b>Subject:</b> %{text}<br>' +
                     '<b>Label:</b> %{customdata[0]}<br>' +
                     '<b>Source:</b> %{customdata[1]}<extra></extra>',
        customdata=viz_df[['label', 'source']].values,
        name='t-SNE'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Email Phishing Dataset Clustering Visualization",
    showlegend=False,
    height=600
)

fig.show()

## 5. Cluster Analysis

print("\n" + "="*50)
print("5. CLUSTER ANALYSIS")
print("="*50)

# Analyze each cluster
for cluster_id in range(N_CLUSTER):
    cluster_data = df[df['cluster'] == cluster_id]
    print(f"\n--- CLUSTER {cluster_id} ---")
    print(f"Size: {len(cluster_data)} samples")
    print(f"Label distribution:")
    print(cluster_data['label'].value_counts())
    print(f"Source distribution:")
    print(cluster_data['source'].value_counts())
    
    # Show sample subjects from this cluster
    print(f"\nSample subjects:")
    sample_subjects = cluster_data['subject'].head(5).tolist()
    for i, subject in enumerate(sample_subjects, 1):
        print(f"  {i}. {subject[:100]}...")

# Create a summary heatmap
cluster_summary = df.groupby(['cluster', 'label']).size().unstack(fill_value=0)
plt.figure(figsize=(10, 6))
sns.heatmap(cluster_summary, annot=True, fmt='d', cmap='Blues')
plt.title('Cluster vs Label Distribution Heatmap')
plt.xlabel('Label')
plt.ylabel('Cluster')
plt.show()

## 6. Sampling and Saving

print("\n" + "="*50)
print("6. SAMPLING AND SAVING")
print("="*50)

def balanced_sample(cluster_data, n_samples, label_col='label', random_state=42):
    """
    Sample data from cluster while maintaining label balance
    """
    if len(cluster_data) <= n_samples:
        return cluster_data
    
    # Get label distribution in the cluster
    label_counts = cluster_data[label_col].value_counts()
    labels = label_counts.index.tolist()
    
    # Calculate proportional samples for each label
    total_cluster_size = len(cluster_data)
    sampled_data_list = []
    
    print(f"    Original label distribution: {dict(label_counts)}")
    
    for i, label in enumerate(labels):
        label_data = cluster_data[cluster_data[label_col] == label]
        
        # Calculate proportional sample size
        if i < len(labels) - 1:  # For all labels except the last one
            label_proportion = len(label_data) / total_cluster_size
            label_sample_size = int(n_samples * label_proportion)
        else:  # For the last label, use remaining samples
            label_sample_size = n_samples - sum(len(df) for df in sampled_data_list)
        
        # Ensure we don't sample more than available
        label_sample_size = min(label_sample_size, len(label_data))
        
        if label_sample_size > 0:
            sampled_label_data = label_data.sample(n=label_sample_size, random_state=random_state)
            sampled_data_list.append(sampled_label_data)
            print(f"    {label}: {len(label_data)} -> {label_sample_size} samples")
    
    # Combine all sampled data
    if sampled_data_list:
        result = pd.concat(sampled_data_list, ignore_index=True)
        # Shuffle the combined data
        result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)
        return result
    else:
        return pd.DataFrame()

def simple_sample(cluster_data, n_samples, random_state=42):
    """
    Simple random sampling without considering label balance
    """
    if len(cluster_data) <= n_samples:
        return cluster_data
    return cluster_data.sample(n=n_samples, random_state=random_state)

# Sample from each cluster and save
sampling_summary = []

for cluster_id in range(N_CLUSTER):
    cluster_data = df[df['cluster'] == cluster_id]
    print(f"\n--- CLUSTER {cluster_id} ---")
    print(f"Cluster size: {len(cluster_data)}")
    
    # Choose sampling method
    if BALANCED_SAMPLING:
        print("Using balanced sampling...")
        sampled_data = balanced_sample(cluster_data, N_SAMPLES_PER_CLUSTER)
    else:
        print("Using simple random sampling...")
        sampled_data = simple_sample(cluster_data, N_SAMPLES_PER_CLUSTER)
    
    print(f"Final sample size: {len(sampled_data)}")
    
    if len(sampled_data) > 0:
        final_label_dist = sampled_data['label'].value_counts()
        print(f"Final label distribution: {dict(final_label_dist)}")
        
        # Store summary info
        sampling_summary.append({
            'cluster_id': cluster_id,
            'original_size': len(cluster_data),
            'sampled_size': len(sampled_data),
            'original_label_dist': dict(cluster_data['label'].value_counts()),
            'sampled_label_dist': dict(final_label_dist)
        })
    
    # Save to file
    output_filename = f"email_phishing_CEAS-08_train_Cluster_{cluster_id + 1}.csv.gz"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    # Remove the temporary columns before saving
    columns_to_save = ['subject', 'body', 'label', 'source', 'cluster']
    sampled_data[columns_to_save].to_csv(output_path, compression='gzip', index=False)
    
    print(f"Saved: {output_path}")

print(f"\nAll cluster files saved to: {OUTPUT_DIR}")

# Create sampling summary visualization
if sampling_summary:
    print("\n" + "="*50)
    print("SAMPLING SUMMARY VISUALIZATION")
    print("="*50)
    
    # Create comparison plots
    fig, axes = plt.subplots(2, N_CLUSTER, figsize=(4*N_CLUSTER, 8))
    if N_CLUSTER == 1:
        axes = axes.reshape(2, 1)
    
    for i, summary in enumerate(sampling_summary):
        cluster_id = summary['cluster_id']
        
        # Original distribution
        orig_labels = list(summary['original_label_dist'].keys())
        orig_counts = list(summary['original_label_dist'].values())
        
        axes[0, i].bar(orig_labels, orig_counts, alpha=0.7, color='skyblue')
        axes[0, i].set_title(f'Cluster {cluster_id}: Original\n(n={summary["original_size"]})')
        axes[0, i].set_ylabel('Count')
        
        # Sampled distribution
        samp_labels = list(summary['sampled_label_dist'].keys())
        samp_counts = list(summary['sampled_label_dist'].values())
        
        axes[1, i].bar(samp_labels, samp_counts, alpha=0.7, color='lightcoral')
        axes[1, i].set_title(f'Cluster {cluster_id}: Sampled\n(n={summary["sampled_size"]})')
        axes[1, i].set_ylabel('Count')
        axes[1, i].set_xlabel('Label')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed sampling summary table
    print("\nDetailed Sampling Summary:")
    print("-" * 80)
    for summary in sampling_summary:
        print(f"Cluster {summary['cluster_id']}:")
        print(f"  Original: {summary['original_size']} samples, {summary['original_label_dist']}")
        print(f"  Sampled:  {summary['sampled_size']} samples, {summary['sampled_label_dist']}")
        
        # Calculate label preservation ratios
        for label in summary['original_label_dist']:
            orig_ratio = summary['original_label_dist'][label] / summary['original_size']
            samp_ratio = summary['sampled_label_dist'].get(label, 0) / summary['sampled_size'] if summary['sampled_size'] > 0 else 0
            print(f"    {label}: {orig_ratio:.3f} -> {samp_ratio:.3f} (ratio preservation: {samp_ratio/orig_ratio:.3f})" if orig_ratio > 0 else f"    {label}: 0 -> 0")
        print()

## 7. Summary Statistics

print("\n" + "="*50)
print("7. SUMMARY")
print("="*50)

print(f"Original dataset size: {len(df)}")
print(f"Number of clusters: {N_CLUSTER}")
print(f"Target samples per cluster: {N_SAMPLES_PER_CLUSTER}")
print(f"Balanced sampling: {BALANCED_SAMPLING}")

if sampling_summary:
    total_sampled = sum(s['sampled_size'] for s in sampling_summary)
    print(f"\nCluster sampling breakdown:")
    for summary in sampling_summary:
        cluster_id = summary['cluster_id']
        print(f"Cluster {cluster_id}: {summary['original_size']} -> {summary['sampled_size']} samples")
    
    print(f"\nTotal samples saved: {total_sampled}")
    
    # Overall label preservation analysis
    if BALANCED_SAMPLING:
        print(f"\nLabel preservation analysis:")
        # Calculate overall label ratios
        original_overall = df['label'].value_counts()
        
        # Combine all sampled data for overall analysis
        all_sampled_labels = []
        for summary in sampling_summary:
            for label, count in summary['sampled_label_dist'].items():
                all_sampled_labels.extend([label] * count)
        
        if all_sampled_labels:
            sampled_overall = pd.Series(all_sampled_labels).value_counts()
            
            print("Overall label distribution comparison:")
            for label in original_overall.index:
                orig_ratio = original_overall[label] / len(df)
                samp_ratio = sampled_overall.get(label, 0) / len(all_sampled_labels)
                preservation = samp_ratio / orig_ratio if orig_ratio > 0 else 0
                print(f"  {label}: {orig_ratio:.3f} -> {samp_ratio:.3f} (preservation: {preservation:.3f})")

print(f"\nSilhouette score: {silhouette_score(embeddings, cluster_labels):.4f}")
print("\nClustering completed successfully!")

# Additional: Save cluster analysis report
print(f"\nSaving cluster analysis report...")
report_path = os.path.join(OUTPUT_DIR, "cluster_analysis_report.txt")
with open(report_path, 'w') as f:
    f.write("Email Phishing Dataset Clustering Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Configuration:\n")
    f.write(f"- Number of clusters: {N_CLUSTER}\n")
    f.write(f"- Samples per cluster: {N_SAMPLES_PER_CLUSTER}\n")
    f.write(f"- Balanced sampling: {BALANCED_SAMPLING}\n")
    f.write(f"- Original dataset size: {len(df)}\n")
    f.write(f"- Silhouette score: {silhouette_score(embeddings, cluster_labels):.4f}\n\n")
    
    if sampling_summary:
        for summary in sampling_summary:
            f.write(f"Cluster {summary['cluster_id']}:\n")
            f.write(f"  Original size: {summary['original_size']}\n")
            f.write(f"  Sampled size: {summary['sampled_size']}\n")
            f.write(f"  Original labels: {summary['original_label_dist']}\n")
            f.write(f"  Sampled labels: {summary['sampled_label_dist']}\n\n")

print(f"Report saved: {report_path}")

Index(['subject', 'body', 'label', 'source'], dtype='object')